In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
#df = pd.read_csv(r"C:\Users\Usuario\Desktop\data_football_ratings.csv")
df = pd.read_csv('data_football_ratings.csv')

In [ ]:
df.head()

,competition,date,match,team,pos,pos_role,player,rater,is_human,original_rating,...,betweenness_centrality,closeness_centrality,flow_centrality,flow_success,betweenness2goals,win,lost,is_home_team,minutesPlayed,game_duration
0,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DC,Dragos Grigore,Kicker,1,3.50,...,0.143055,0.603571,0.304348,0.000000,0.0,0,1,0,90,90
1,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DC,Dragos Grigore,WhoScored,0,6.56,...,0.143055,0.603571,0.304348,0.000000,0.0,0,1,0,90,90
2,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,DF,DC,Dragos Grigore,SofaScore,0,6.70,...,0.143055,0.603571,0.304348,0.000000,0.0,0,1,0,90,90
3,Euro 2016,10/06/2016,"France - Romania, 2 - 1",France,Sub,Sub,Anthony Martial,WhoScored,0,6.19,...,0.051556,0.524845,0.041096,0.166667,0.0,1,0,1,13,90
4,Euro 2016,10/06/2016,"France - Romania, 2 - 1",Romania,MF,DMC,Mihai Pintilii,Kicker,1,3.50,...,0.333284,0.710084,0.347826,0.675075,0.0,0,1,0,90,90


In [3]:
# Mantener solo las filas donde is_human sea diferente de 1
df = df[df["is_human"] != 1]

In [4]:
df2 = df[df["rater"]=='WhoScored']
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 1 to 50651
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             21354 non-null  object 
 1   date                    21354 non-null  object 
 2   match                   21354 non-null  object 
 3   team                    21354 non-null  object 
 4   pos                     21354 non-null  object 
 5   pos_role                21354 non-null  object 
 6   player                  21354 non-null  object 
 7   rater                   21354 non-null  object 
 8   is_human                21354 non-null  int64  
 9   original_rating         21354 non-null  float64
 10  goals                   21354 non-null  int64  
 11  assists                 21354 non-null  int64  
 12  shots_ontarget          21354 non-null  int64  
 13  shots_offtarget         21354 non-null  int64  
 14  shotsblocked            21354 non-null  int

In [5]:
#Filtrar los datos para analizar ÚNICAMENTE a los  ('MF')
df_medios = df2[df2['pos'] == 'MF'].copy()



In [6]:
df_medios['pos'].value_counts()

,count
pos,
MF,7206


In [7]:
cols_to_drop = [
    "competition", "date", "match", "team", "player", "rater", "is_human",
    "degree_centrality", "betweenness_centrality", "closeness_centrality",
    "flow_centrality", "flow_success", "betweenness2goals","pos_role","pos"
]


In [8]:
df_medios = df_medios.drop(columns= cols_to_drop)

In [9]:
df_medios.info ()

<class 'pandas.core.frame.DataFrame'>
Index: 7206 entries, 5 to 50648
Data columns (total 48 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   original_rating   7206 non-null   float64
 1   goals             7206 non-null   int64  
 2   assists           7206 non-null   int64  
 3   shots_ontarget    7206 non-null   int64  
 4   shots_offtarget   7206 non-null   int64  
 5   shotsblocked      7206 non-null   int64  
 6   chances2score     7206 non-null   int64  
 7   drib_success      7206 non-null   int64  
 8   drib_unsuccess    7206 non-null   int64  
 9   keypasses         7206 non-null   int64  
 10  touches           7206 non-null   int64  
 11  passes_acc        7206 non-null   int64  
 12  passes_inacc      7206 non-null   int64  
 13  crosses_acc       7206 non-null   int64  
 14  crosses_inacc     7206 non-null   int64  
 15  lballs_acc        7206 non-null   int64  
 16  lballs_inacc      7206 non-null   int64  
 17 

In [ ]:
corrs = df_medios.corr()['original_rating'].sort_values(ascending=False)
print (corrs)

original_rating     1.000000
goals               0.513103
win                 0.463809
shots_ontarget      0.428021
touches             0.387518
assists             0.371051
grduels_w           0.349561
minutesPlayed       0.288077
passes_acc          0.262009
drib_success        0.259593
poss_lost           0.247938
countattack         0.237818
keypasses           0.231113
chances2score       0.227989
tballs_acc          0.204012
passes_inacc        0.195717
lballs_acc          0.182206
tackles             0.181434
crosses_acc         0.175435
aerials_w           0.163444
wasfouled           0.158157
shots_offtarget     0.139052
tballs_inacc        0.137405
interceptions       0.135095
is_home_team        0.123818
lballs_inacc        0.120963
clearances          0.102487
shotsblocked        0.096614
crosses_inacc       0.075731
grduels_l           0.047009
drib_unsuccess      0.045426
stop_shots          0.038402
offsides            0.019893
missed_penalties    0.010398
aerials_l     

In [14]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Separar X y y
X = df_medios.drop(columns=["original_rating"])
y = df_medios["original_rating"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [15]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.7 MB/s eta 0:00:00


In [16]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

# Entrenar modelo
catboost = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost.fit(X_train, y_train)

# Predecir en test
y_pred = catboost.predict(X_test)

# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7306
CatBoost RMSE: 0.3730


In [ ]:
import pandas as pd

# Obtener importancias
importancias = catboost.get_feature_importance(prettified=True)

# Si quieres sólo las más importantes, ordena y selecciona las top N
top_features = importancias.sort_values('Importances', ascending=False)

print(top_features)  # Muestra todas ordenadas

# Opcional: mostrar sólo las 10 primeras
print(top_features.head(20))

          Feature Id  Importances
0              goals    21.574858
1            assists     9.825256
2            touches     9.645295
3               lost     8.830085
4                win     8.407925
5          grduels_w     5.073073
6          aerials_w     3.855223
7       drib_success     3.820016
8     shots_ontarget     3.713457
9          keypasses     3.123256
10           tackles     2.744445
11     interceptions     1.990739
12     minutesPlayed     1.810281
13        clearances     1.738032
14        passes_acc     1.318676
15         aerials_l     1.075890
16         poss_lost     0.977475
17            ycards     0.894571
18        stop_shots     0.887250
19        lballs_acc     0.824242
20       crosses_acc     0.736581
21     crosses_inacc     0.697016
22      passes_inacc     0.696265
23      dangmistakes     0.597196
24         grduels_l     0.552232
25       countattack     0.536471
26         wasfouled     0.507167
27      lballs_inacc     0.445294
28        tbal

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Lista de modelos a evaluar
modelos = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    "SVR": SVR(kernel="rbf", C=10, epsilon=0.1),
    "KNN": KNeighborsRegressor(n_neighbors=5)
}

# Entrenar y evaluar cada modelo
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"{nombre} -> R²: {r2:.4f}, RMSE: {rmse:.4f}")

LinearRegression -> R²: 0.7383, RMSE: 0.3676
Ridge -> R²: 0.7382, RMSE: 0.3677
Lasso -> R²: 0.7144, RMSE: 0.3840
ElasticNet -> R²: 0.7232, RMSE: 0.3781
RandomForest -> R²: 0.6738, RMSE: 0.4105
GradientBoosting -> R²: 0.7159, RMSE: 0.3830
SVR -> R²: 0.6766, RMSE: 0.4087
KNN -> R²: 0.1946, RMSE: 0.6449


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Entrenar modelo Ridge (puedes ajustar alpha según prefieras)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)

# Predecir en test
y_pred = ridge.predict(X_test)

# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Ridge R²: {r2:.4f}")
print(f"Ridge RMSE: {rmse:.4f}")

Ridge R²: 0.7382
Ridge RMSE: 0.3677


In [ ]:
# Supongamos que tus nombres de columnas están en X_train.columns
importancias = pd.Series(ridge.coef_, index=X_train.columns)

# Ordenar de mayor a menor peso absoluto
importancias_ordenadas = importancias.abs().sort_values(ascending=False)

print("Importancia (coeficientes absolutos) de cada variable:")
print(importancias_ordenadas)

# Opcional: mostrar solo las 10 variables más influyentes
print(importancias_ordenadas.head(10))

Importancia (coeficientes absolutos) de cada variable:
goals               0.776551
rcards              0.681732
assists             0.571242
missed_penalties    0.331513
lost                0.273369
owngoals            0.236586
win                 0.232019
saves_otb           0.155084
shots_ontarget      0.130395
keypasses           0.126389
ycards              0.112165
stop_shots          0.102905
dangmistakes        0.086139
drib_success        0.079975
aerials_w           0.060812
tackles             0.047868
clearances          0.044443
offsides            0.037129
interceptions       0.028708
grduels_w           0.027970
tballs_acc          0.027006
aerials_l           0.023355
crosses_acc         0.022224
drib_unsuccess      0.019036
chances2score       0.015507
crosses_inacc       0.015072
poss_lost           0.014733
dribbled_past       0.013962
touches             0.010423
is_home_team        0.009654
wasfouled           0.008472
countattack         0.006678
lballs_acc       

In [17]:
import pandas as pd

# Suponiendo que y_test es un array/pd.Series y y_pred es un array
comparacion = pd.DataFrame({
    'Original': y_test,
    'Prediccion': y_pred
})

# Mostrar las primeras filas para ver los valores
print(comparacion.head())

# Si quieres también ver el error absoluto de cada predicción:
comparacion['Error absoluto'] = (comparacion['Original'] - comparacion['Prediccion']).abs()
print(comparacion.head(20))

       Original  Prediccion
21355      7.39    7.047559
11266      6.58    6.415403
42694      7.54    7.901725
1611       6.42    6.927354
2155       7.33    7.220559
       Original  Prediccion  Error absoluto
21355      7.39    7.047559        0.342441
11266      6.58    6.415403        0.164597
42694      7.54    7.901725        0.361725
1611       6.42    6.927354        0.507354
2155       7.33    7.220559        0.109441
18595      7.64    7.383780        0.256220
43124      7.65    7.347193        0.302807
47320      6.72    6.648511        0.071489
18736      6.52    6.311568        0.208432
6063       6.50    7.146419        0.646419
27772      6.96    6.812111        0.147889
16207      7.17    6.984264        0.185736
24358      6.33    6.337756        0.007756
45622      7.99    7.800681        0.189319
13824      5.96    6.186610        0.226610
37904      6.52    6.168889        0.351111
43404      6.17    6.390061        0.220061
1693       8.68    8.434481        0.245

In [10]:
import numpy as np

# Asumiendo que tienes win (1/0) y lost (1/0)
# Empate donde ni win ni lost son 1
df_medios['result'] = np.where(df_medios['win'] == 1, 'victory',
                 np.where(df_medios['lost'] == 1, 'defeat', 'draw'))

# Luego elimina las columnas originales
df_medios = df_medios.drop(columns=['win', 'lost'])

In [11]:
df_medios['duelos_ganados'] = df_medios['aerials_w'] + df_medios['grduels_w']
df_medios = df_medios.drop(columns=['aerials_w', 'grduels_w'])

In [12]:
cols = [
    "goals", "touches", "assists", "result", "shots_ontarget",
    "duelos_ganados", "drib_success", "poss_lost", "passes_acc",
    "minutesPlayed", "interceptions", "tackles", "keypasses",
    "ycards", "rcards", "original_rating"
]

df_medios = df_medios[cols].copy()


In [13]:
df_medios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7206 entries, 5 to 50648
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   goals            7206 non-null   int64  
 1   touches          7206 non-null   int64  
 2   assists          7206 non-null   int64  
 3   result           7206 non-null   object 
 4   shots_ontarget   7206 non-null   int64  
 5   duelos_ganados   7206 non-null   int64  
 6   drib_success     7206 non-null   int64  
 7   poss_lost        7206 non-null   int64  
 8   passes_acc       7206 non-null   int64  
 9   minutesPlayed    7206 non-null   int64  
 10  interceptions    7206 non-null   int64  
 11  tackles          7206 non-null   int64  
 12  keypasses        7206 non-null   int64  
 13  ycards           7206 non-null   int64  
 14  rcards           7206 non-null   int64  
 15  original_rating  7206 non-null   float64
dtypes: float64(1), int64(14), object(1)
memory usage: 957.0+ KB


In [14]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Separar X e y
X = df_medios.drop(columns=['original_rating'])
y = df_medios['original_rating']

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definir variables categóricas si las hay (ajustar según tu dataset)
cat_features = ['result']  # ejemplo

# Obtener índices de variables categóricas
cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features if col in X_train.columns]

# Inicializar y entrenar modelo CatBoost
model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=4, random_seed=42, verbose=0,l2_leaf_reg= 1)
model.fit(X_train, y_train, cat_features=cat_features_idx)

# Predecir y evaluar
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7100
CatBoost RMSE: 0.3870


In [23]:
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV

# Definir una grilla pequeña para búsqueda rápida
param_grid = {
    'iterations': [200, 500],          # dos valores razonables
    'learning_rate': [0.03, 0.05],     # dos valores rápidos
    'depth': [4, 6],                   # dos valores poco costosos
    'l2_leaf_reg': [1, 3]              # opcional, regularización bajita
}

# Instanciamos el modelo base
catboost = CatBoostRegressor(random_seed=42, verbose=0)

# GridSearchCV con menos folds para que sea más rápido
grid_search = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring='r2',
    cv=3,                             # menos folds = más rápido
    n_jobs=-1,                        # usa todos los núcleos
    refit=True
)

# Ejecutar grid search
grid_search.fit(X_train, y_train, cat_features=cat_features_idx)

# Resultados
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor R² CV:", grid_search.best_score_)

# Evaluamos el mejor modelo en el test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"CatBoost reajustado R² test: {r2:.4f}")
print(f"CatBoost reajustado RMSE: {rmse:.4f}")


Mejores parámetros: {'depth': 4, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.05}
Mejor R² CV: 0.7125249927140924
CatBoost reajustado R² test: 0.7100
CatBoost reajustado RMSE: 0.3870


In [18]:
import pandas as pd

# Caso sintético con columnas seleccionadas
nuevo_jugador = pd.DataFrame([{
    'goals': 0,
    'touches': 10,
    'assists': 0,
    'result': "defeat",
    'shots_ontarget': 1,
    'duelos_ganados': 4,
    'drib_success': 1,
    'poss_lost': 2,
    'passes_acc': 12,
    'minutesPlayed': 40,
    'interceptions': 4,
    'tackles': 10,
    'keypasses': 1,
    'ycards': 0,
    'rcards': 0
}])

In [16]:
def predecir_limitado(model, X, min_val=0, max_val=10):
    y_pred = model.predict(X)
    return np.clip(y_pred, min_val, max_val)

In [19]:
predecir_limitado(model, nuevo_jugador)

array([6.93977594])

In [30]:
df_medios.head(10)

,original_rating,goals,assists,shots_ontarget,drib_success,keypasses,touches,passes_acc,poss_lost,interceptions,tackles,ycards,rcards,minutesPlayed,result,duelos_ganados
5,6.58,0,0,0,2,0,60,31,14,7,3,0,0,90,defeat,6
14,7.06,0,0,1,2,0,74,54,9,3,2,0,0,90,victory,8
20,7.67,0,1,0,2,0,106,76,10,9,6,0,0,90,victory,8
26,7.26,0,0,0,1,3,33,16,12,1,1,0,0,72,defeat,6
32,6.74,1,0,1,0,0,33,18,11,2,0,0,0,90,defeat,1
41,5.85,0,0,0,0,0,24,7,10,0,2,1,0,82,defeat,4
44,6.12,0,0,0,0,0,33,17,11,8,0,0,0,90,defeat,1
65,6.62,0,0,1,1,0,77,50,13,2,1,0,0,77,victory,7
82,8.13,0,1,1,1,1,60,40,10,1,1,0,0,88,victory,5
92,6.51,0,0,0,0,0,44,20,9,1,1,0,0,90,victory,3


In [20]:
model.save_model("catboost_medios.cbm")
